# RAAI-SGC 2.0 — Colab T4: LLM + optional QLoRA

Use this notebook with your **existing unified project ZIP**, not a second project. We first retrieve evidence from `data/processed/master.csv`, then ask a pretrained instruction-tuned model to answer. **Fine-tuning remains disabled until genuinely approved training examples exist.** A Colab T4 may be unavailable due to quota.

**Before running:** Colab menu → Runtime → Change runtime type → **T4 GPU**. Upload your `RAAI_SGC_2_COMPLETE_PROJECT.zip` when prompted.

In [ ]:
import torch, subprocess
print('Torch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then rerun.')
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout[:2500])

## Install runtime dependencies
We reuse Colab's PyTorch installation. The optional 4-bit libraries support a T4; do not enable bfloat16 (`bf16`) on a T4.

In [ ]:
%pip -q install 'transformers>=4.48,<5' 'accelerate>=1,<2' 'scikit-learn>=1.3,<2' 'peft>=0.15,<1' 'bitsandbytes>=0.45' 'datasets>=3,<5'
import transformers, accelerate
print('Transformers:',transformers.__version__)

## Import the existing project ZIP
Upload the full ZIP from your Ubuntu `~/Downloads` folder. If you changed code locally, ZIP the **current** folder first so Colab gets those changes.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys
uploaded=files.upload()
zip_names=[n for n in uploaded if n.lower().endswith('.zip')]
assert len(zip_names)==1, 'Upload exactly one existing unified project ZIP'
zip_path=Path('/content')/zip_names[0]
with zipfile.ZipFile(zip_path) as z:
    bad=[n for n in z.namelist() if Path(n).is_absolute() or '..' in Path(n).parts]
    if bad: raise ValueError('Unsafe zip members: '+str(bad[:3]))
    z.extractall('/content')
folders=[p for p in Path('/content').rglob('data/processed/master.csv') if 'RAAI_SGC_2_COMPLETE_PROJECT' in p.as_posix()]
assert folders,'master.csv not found inside uploaded ZIP'
ROOT=folders[0].parents[2]
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
print('Project folder:',ROOT)
print('Master CSV:',ROOT/'data/processed/master.csv')

## Verify the real dataset and retrieval
Q&A drafts are not training examples. The source-backed retrieval works before the LLM is loaded.

In [ ]:
from src.hybrid import load_data, HybridRetriever
sources,edges,qa_draft_count=load_data(ROOT/'data/processed/master.csv')
retriever=HybridRetriever(sources,edges,backend='tfidf')
print('Sources:',len(sources),'Q&A drafts (NOT approved):',qa_draft_count)
question='What is Kubernetes?'
results,diagnostics=retriever.search(question,mode='hybrid',top_k=3)
print('Graph active:',diagnostics.get('graph_activated'))
for r in results: print(r['source_title'],r['source_url'])

## Load the generative LLM on the T4
A modest starting model is `Qwen/Qwen2.5-1.5B-Instruct` (Apache-2.0 per its model card). It is a pretrained model, **not** yet fine-tuned on your data. The initial model download requires internet access.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_ID='Qwen/Qwen2.5-1.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
quant_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=quant_config,device_map='auto')
model.eval()
print('Loaded:',MODEL_ID)

## Ask an evidence-grounded question
This uses your **real corpus** and produces a generative answer with numbered evidence passages. References are retrieved, not independently fact-checked.

In [ ]:
def ask(question,mode='hybrid',top_k=3):
    retrieved,diag=retriever.search(question,mode=mode,top_k=top_k)
    context='\n\n'.join(f"[{i}] {r['source_title']} ({r['source_url']})\n{r['source_text'][:950]}" for i,r in enumerate(retrieved,1))
    messages=[
        {'role':'system','content':'You are an IT technical assistant. Answer using only supplied evidence. If evidence is insufficient, say so. Cite source numbers [1], [2].'},
        {'role':'user','content':f'Evidence:\n{context}\n\nQuestion: {question}'}
    ]
    inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output=model.generate(inputs,max_new_tokens=180,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    text=tokenizer.decode(output[0][inputs.shape[1]:],skip_special_tokens=True)
    print('QUESTION:',question,'\nANSWER:',text,'\nSOURCES:')
    for i,r in enumerate(retrieved,1): print(f"[{i}] {r['source_title']} — {r['source_url']}")
    print('Graph activated:',diag.get('graph_activated'))
    return text

ask('What is Kubernetes?')

## Optional: Try another retrieval backend
The default example uses TF-IDF. For semantic embeddings, install `sentence-transformers` and recreate the retriever with `backend="semantic"`. This downloads additional model weights and may use GPU memory; release the LLM first if you encounter CUDA out-of-memory errors.

In [ ]:
# Optional — do not run if GPU memory is low:
# %pip -q install 'sentence-transformers>=3,<6'
# retriever=HybridRetriever(sources,edges,backend='semantic')
# ask('How do authentication and authorization differ?')

## QLoRA training gate (deliberately stops when approvals are absent)
The delivered dataset has **zero approved supervised Q&A records**. Review source rights, answer accuracy, and train/test separation first. Run this cell to confirm that the exporter will not silently train on draft material.

In [ ]:
import subprocess,sys
prepare=subprocess.run([sys.executable,'-m','training.prepare_sft'],cwd=str(ROOT),capture_output=True,text=True)
print(prepare.stdout or prepare.stderr)
if prepare.returncode != 0: raise RuntimeError('Approval exporter failed')
train_file=ROOT/'data/training/sft_train.jsonl'
rows=[x for x in train_file.read_text(encoding='utf-8').splitlines() if x.strip()] if train_file.exists() else []
print('Reviewed training records:',len(rows))
print('Eligible for smoke training:',len(rows)>=20)
if not rows:
    print('TRAINING IS BLOCKED: approval decisions are not yet recorded. This is expected.')

## Optional: QLoRA fine-tuning (approved rows only)
Run **only after** approved examples have been added to `data/review/APPROVED_QA_ONLY.csv` in the unified folder, the project ZIP has been updated, and the earlier preparation cell confirms at least 20 approved train examples. A minimum of 20 supports only a technical smoke test, **not** a scientifically persuasive fine-tuning experiment. Free GPU runtime limits vary. Do not use gold evaluation questions for training.

In [ ]:
# Intentionally gated. Does not start training with zero approved data.
import gc
if len(rows)<20:
    print(f'SKIPPED: {len(rows)} approved examples. Add reviewed training examples and rerun the preparation cell.')
else:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    cmd=[sys.executable,'-m','training.train_qlora_colab','--model',MODEL_ID,'--data',str(train_file),'--output',str(ROOT/'models/checkpoints/qlora_colab'),'--epochs','1','--max-steps','30']
    subprocess.run(cmd,cwd=str(ROOT),check=True)

## Export your adapter and save the notebook
The LoRA adapter is a small set of weights, **not a standalone base model**. To use it later you must load the same compatible pretrained base model and the adapter, or merge/export in a separate supported workflow.

In [ ]:
from google.colab import files
import shutil
adapter=ROOT/'models/checkpoints/qlora_colab'
if (adapter/'adapter_config.json').exists():
    archive=shutil.make_archive('/content/RAAI_SGC_2_QLORA_ADAPTER','zip',root_dir=adapter)
    files.download(archive)
else:
    print('No trained adapter yet. First complete approved-data review and GPU training.')

### Project integrity and reporting
- Baselines: identical retrieved corpus and evaluation set for vector-only, unweighted graph, and weighted hybrid.
- Keep independent gold evaluation questions **out** of QLoRA training.
- Record model ID, revision, package versions, training hyperparameters, GPU type, seeds, and measured outcomes.
- A T4 notebook finishing successfully does not prove graph augmentation improves answers; run controlled evaluation before writing paper results.
- For a live demonstration with Colab GPU, run the Gradio app in the Colab runtime or serve an authenticated model endpoint. The unchanged Ubuntu app does not automatically access the remote T4 GPU.